# Sprint 7 · Reto «Precio y cancelación» · Equipo 5 Green

**Equipo:** Cristina, Miguel, Fran, Andri y Anzhela · **Cliente:** TurisData (datos sintéticos de un hotel).

Este cuaderno contiene **dos modelos** y la traducción de sus métricas a **euros**:

1. **Precio:** tarifa de referencia por noche según tipo de habitación y época (regresión → un número).
2. **Cancelación:** aviso de riesgo para decidir cuándo hacer *overbooking* (clasificación → sí/no), con el **umbral que menos cuesta**.

**Cómo ejecutarlo:** *Entorno de ejecución → Reiniciar y ejecutar todo*. Todo se calcula con `SEMILLA = 42`.
El CSV se descarga solo del repositorio (o se lee de `data/` si ya está).

**Regla del test:** el conjunto de test **no se toca** hasta el apartado 4.6 (evaluación final). Todo lo que se elige
(modelo, umbral, k de k-NN) se decide con validación cruzada sobre *train*.

| Apartado | Qué hace |
|---|---|
| 1 | Preparación, versiones y datos |
| 2 | Variables usadas y excluidas (sin fuga) |
| 3 | Modelo de **precio** (baseline, lineal, interacciones) |
| 4 | Modelo de **cancelación** (baseline, logística, k-NN), costes, umbral, sensibilidad, test final e IA responsable |
| 5 | Cifras del informe (qué celda produce cada una) |

## 1 · Preparación, versiones y datos

In [1]:
#@title 1.1 · ▶ Ejecuta esta celda primero
import os, sys, urllib.request, json, warnings

BASE_URL = "https://raw.githubusercontent.com/cCoolBoox/IFCD107-alumnado/main/datos/"

def dato(nombre):
    """Devuelve la ruta local del CSV (carpetas data/ o datos/ del equipo) y lo descarga si no está."""
    for carpeta in ("data", "../data", "datos", "../datos", "../../datos", "../../../datos"):
        ruta = os.path.join(carpeta, nombre)
        if os.path.exists(ruta):
            return ruta
    os.makedirs("datos", exist_ok=True)
    ruta = os.path.join("datos", nombre)
    urllib.request.urlretrieve(BASE_URL + nombre, ruta)
    return ruta

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split, cross_val_score, cross_val_predict, StratifiedKFold
from sklearn.dummy import DummyRegressor, DummyClassifier
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import PolynomialFeatures, StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.metrics import (mean_absolute_error, mean_squared_error, r2_score, confusion_matrix,
                             ConfusionMatrixDisplay, roc_auc_score, average_precision_score,
                             roc_curve, precision_recall_curve, precision_score, recall_score, accuracy_score)

warnings.filterwarnings("ignore", category=UserWarning)
SEMILLA = 42
np.random.seed(SEMILLA)
sns.set_theme(style="whitegrid")

print("Python", sys.version.split()[0], "· scikit-learn", sklearn.__version__,
      "· pandas", pd.__version__, "· numpy", np.__version__)

df = pd.read_csv(dato("reservas_turisdata.csv"), parse_dates=["fecha_reserva", "fecha_llegada"])
df["mes_llegada"] = df["fecha_llegada"].dt.month
df["temporada_alta"] = df["mes_llegada"].isin([1, 2, 3, 7, 8, 12]).astype(int)   # misma regla que S07_01
print(df.shape, "· tasa de cancelación:", round(df["cancelada"].mean(), 3))
print("Precio medio por noche:", round(df["precio_noche"].mean(), 2), "€ · noches medias:", round(df["noches"].mean(), 2))

Python 3.12.3 · scikit-learn 1.9.1 · pandas 3.0.6 · numpy 2.5.4
(6000, 20) · tasa de cancelación: 0.459
Precio medio por noche: 102.87 € · noches medias: 5.52


## 2 · Variables usadas y excluidas (sin fuga de datos)

Criterio: una variable solo entra si **se conoce en el momento de reservar**.

| Variable | ¿Precio? | ¿Cancelación? | Razón |
|---|---|---|---|
| `tipo_habitacion`, `temporada_alta` (sale del mes de llegada) | ✅ | ✅ | Se eligen/conocen al reservar. La tarifa de referencia depende solo de estas dos. |
| `antelacion_dias`, `noches`, `adultos`, `ninos`, `regimen`, `canal`, `pais_origen`, `tarifa`, `deposito`, `cliente_repetidor`, `cancelaciones_previas` | ❌ | ✅ | Se conocen al reservar. No se usan para el precio porque el cliente pidió una tarifa **por tipo de habitación y época**. |
| `precio_noche` | (objetivo) | ✅ | Se conoce al reservar. |
| `peticiones_especiales` | ❌ | ✅ (con aviso) | Puede registrarse **después** de reservar: se comprueba en 4.2 cuánto pesa; si el modelo no cambia sin ella, el aviso es menor. |
| `cancelada` | ❌ **nunca** | (objetivo) | Es el desenlace. No se usa para predecir el precio. |
| `id_reserva`, `fecha_reserva`, `fecha_llegada` | ❌ | ❌ | Identificador y fechas crudas. Solo se usa `fecha_llegada` para derivar el mes/temporada. |

In [2]:
# --- PRECIO: X mínima y explicable (tipo de habitación + temporada)
X_precio = pd.get_dummies(df[["tipo_habitacion"]], drop_first=True).astype(int)
X_precio["temporada_alta"] = df["temporada_alta"]
y_precio = df["precio_noche"]

# --- CANCELACIÓN: todo lo conocido al reservar
columnas_num = ["antelacion_dias", "noches", "adultos", "ninos", "precio_noche",
                "peticiones_especiales", "cliente_repetidor", "cancelaciones_previas", "temporada_alta"]
columnas_cat = ["tipo_habitacion", "regimen", "canal", "pais_origen", "tarifa", "deposito"]
X_canc, y_canc = df[columnas_num + columnas_cat], df["cancelada"]

# Comprobaciones anti-fuga
assert "cancelada" not in X_precio.columns and "precio_noche" not in X_precio.columns
assert "cancelada" not in X_canc.columns
print("X_precio:", X_precio.shape, "· y_precio:", y_precio.shape)
print("X_canc:  ", X_canc.shape, "· y_canc:  ", y_canc.shape)

X_precio: (6000, 4) · y_precio: (6000,)
X_canc:   (6000, 15) · y_canc:   (6000,)


## 3 · Modelo de PRECIO (regresión)

Tres modelos como máximo: **baseline** (precio medio de train), **regresión lineal** y **lineal con interacciones** (la mejora).

In [3]:
Xp_tr, Xp_te, yp_tr, yp_te = train_test_split(X_precio, y_precio, test_size=0.2, random_state=SEMILLA)

base_p = DummyRegressor(strategy="mean").fit(Xp_tr, yp_tr)
lin_p = LinearRegression().fit(Xp_tr, yp_tr)
inter_p = make_pipeline(PolynomialFeatures(degree=2, interaction_only=True, include_bias=False),
                        LinearRegression()).fit(Xp_tr, yp_tr)

def metricas_reg(modelo):
    p = modelo.predict(Xp_te)
    return {"MAE (€)": mean_absolute_error(yp_te, p),
            "RMSE (€)": mean_squared_error(yp_te, p) ** 0.5,
            "R²": r2_score(yp_te, p)}

tabla_precio = pd.DataFrame({"baseline (precio medio)": metricas_reg(base_p),
                             "regresión lineal": metricas_reg(lin_p),
                             "lineal + interacciones": metricas_reg(inter_p)}).T.round(3)
tabla_precio

,MAE (€),RMSE (€),R²
baseline (precio medio),25.564,33.328,-0.002
regresión lineal,7.239,9.704,0.915
lineal + interacciones,6.758,8.936,0.928


In [4]:
# ¿Compensa la complejidad? Validación cruzada (solo con train) de los dos modelos con variables
r2_cv_lin = cross_val_score(LinearRegression(), Xp_tr, yp_tr, cv=5, scoring="r2").mean()
r2_cv_int = cross_val_score(make_pipeline(PolynomialFeatures(2, interaction_only=True, include_bias=False),
                                          LinearRegression()), Xp_tr, yp_tr, cv=5, scoring="r2").mean()
mejora_mae_precio = tabla_precio.loc["regresión lineal", "MAE (€)"] - tabla_precio.loc["lineal + interacciones", "MAE (€)"]
print(f"R² CV (train): lineal {r2_cv_lin:.3f} · con interacciones {r2_cv_int:.3f}")
print(f"Mejora del MAE en test con interacciones: {mejora_mae_precio:.2f} € por noche")

# Coeficientes de la regresión lineal en euros
coefs = pd.Series(lin_p.coef_, index=Xp_tr.columns).round(1)
print("\nPrecio base (estándar, temporada baja):", round(lin_p.intercept_, 1), "€")
print(coefs.to_string())

R² CV (train): lineal 0.926 · con interacciones 0.934
Mejora del MAE en test con interacciones: 0.48 € por noche

Precio base (estándar, temporada baja): 67.3 €
tipo_habitacion_familiar     45.0
tipo_habitacion_suite       101.2
tipo_habitacion_superior     28.4
temporada_alta               22.4


In [5]:
# Tarifa de referencia para el gerente: precio previsto por tipo de habitación y temporada (modelo con interacciones)
tipos = ["estandar", "superior", "familiar", "suite"]
filas = []
for t in tipos:
    for alta in (0, 1):
        fila = pd.DataFrame(0, index=[0], columns=X_precio.columns)
        if f"tipo_habitacion_{t}" in fila.columns:
            fila[f"tipo_habitacion_{t}"] = 1
        fila["temporada_alta"] = alta
        filas.append({"tipo": t, "temporada": "alta" if alta else "baja", "precio_previsto": float(inter_p.predict(fila)[0])})
tarifa_ref = pd.DataFrame(filas).pivot(index="tipo", columns="temporada", values="precio_previsto").loc[tipos].round(0)
tarifa_ref["recargo_alta (€)"] = (tarifa_ref["alta"] - tarifa_ref["baja"]).round(0)
tarifa_ref

temporada,alta,baja,recargo_alta (€)
tipo,,,
estandar,87.0,70.0,17.0
superior,119.0,95.0,24.0
familiar,137.0,110.0,27.0
suite,198.0,161.0,37.0


In [6]:
fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
pred_i = inter_p.predict(Xp_te)
ejes[0].scatter(pred_i, yp_te - pred_i, s=10, alpha=0.4, color="#2a6f97")
ejes[0].axhline(0, color="black", lw=1)
ejes[0].set(title="Residuos frente a precio previsto (test)", xlabel="Precio previsto (€)", ylabel="Error (€)")
tarifa_ref[["baja", "alta"]].plot(kind="bar", ax=ejes[1], color=["#8ecae6", "#2a6f97"], rot=0)
ejes[1].set(title="Tarifa de referencia por tipo y temporada", xlabel="Tipo de habitación", ylabel="Precio por noche (€)")
plt.tight_layout(); plt.show()

## 4 · Modelo de CANCELACIÓN (clasificación) y decisión en euros

### 4.1 · Partición y modelos

`train` (80 %) y `test` (20 %), estratificado. **Test no se toca hasta 4.6.**

In [7]:
Xc_tr, Xc_te, yc_tr, yc_te = train_test_split(X_canc, y_canc, test_size=0.2, random_state=SEMILLA, stratify=y_canc)
print("train:", Xc_tr.shape, "· test:", Xc_te.shape, "· tasa cancelación train:", round(yc_tr.mean(), 3))

def preprocesado(num=columnas_num, cat=columnas_cat):
    return ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("esc", StandardScaler())]), num),
        ("cat", Pipeline([("imp", SimpleImputer(strategy="most_frequent")),
                          ("oh", OneHotEncoder(handle_unknown="ignore"))]), cat),
    ])

# Referencia: «siempre no cancela»
base_c = DummyClassifier(strategy="constant", constant=0)
# Modelo 1: regresión logística
logit = Pipeline([("prep", preprocesado()), ("clf", LogisticRegression(max_iter=1000))])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEMILLA)

train: (4800, 15) · test: (1200, 15) · tasa cancelación train: 0.459


### 4.2 · Comparar modelos con validación cruzada (solo train)

Se compara la referencia, la logística y un segundo modelo (**k-NN**, nivel extra). Máximo 3 modelos.
Primero se elige `k` de k-NN con validación cruzada sobre *train*.

In [8]:
ks = [15, 31, 61, 101, 201]
auc_k = {k: cross_val_score(Pipeline([("prep", preprocesado()), ("clf", KNeighborsClassifier(n_neighbors=k))]),
                            Xc_tr, yc_tr, cv=cv, scoring="roc_auc").mean() for k in ks}
mejor_k = max(auc_k, key=auc_k.get)
print("AUC CV por k:", {k: round(v, 3) for k, v in auc_k.items()}, "→ mejor k =", mejor_k)
knn = Pipeline([("prep", preprocesado()), ("clf", KNeighborsClassifier(n_neighbors=mejor_k))])

modelos_c = {"baseline (siempre no cancela)": base_c, "regresión logística": logit, f"k-NN (k={mejor_k})": knn}
proba_cv_modelos, filas = {}, {}
for nombre, m in modelos_c.items():
    p = cross_val_predict(m, Xc_tr, yc_tr, cv=cv, method="predict_proba")[:, 1]
    proba_cv_modelos[nombre] = p
    filas[nombre] = {"AUC (CV)": roc_auc_score(yc_tr, p), "AP (CV)": average_precision_score(yc_tr, p)}
tabla_cv_clasif = pd.DataFrame(filas).T.round(3)
tabla_cv_clasif

AUC CV por k: {15: np.float64(0.685), 31: np.float64(0.707), 61: np.float64(0.721), 101: np.float64(0.73), 201: np.float64(0.734)} → mejor k = 201


,AUC (CV),AP (CV)
baseline (siempre no cancela),0.500,0.459
regresión logística,0.758,0.719
k-NN (k=201),0.734,0.689


In [9]:
nombre_logit = "regresión logística"
proba_cv = proba_cv_modelos[nombre_logit]          # probabilidades de validación cruzada de la logística
nombre_knn = f"k-NN (k={mejor_k})"

fig, ejes = plt.subplots(1, 2, figsize=(11, 4.5))
for nombre in [nombre_logit, nombre_knn]:
    p = proba_cv_modelos[nombre]
    fpr, tpr, _ = roc_curve(yc_tr, p); pr, rc, _ = precision_recall_curve(yc_tr, p)
    ejes[0].plot(fpr, tpr, lw=2, label=f"{nombre} (AUC {roc_auc_score(yc_tr, p):.2f})")
    ejes[1].plot(rc, pr, lw=2, label=f"{nombre} (AP {average_precision_score(yc_tr, p):.2f})")
ejes[0].plot([0, 1], [0, 1], "--", color="gray", label="Azar")
ejes[0].set(title="Curva ROC (validación cruzada, train)", xlabel="Falsas alarmas (FPR)", ylabel="Cancelaciones detectadas (recall)")
ejes[1].axhline(yc_tr.mean(), ls="--", color="gray", label=f"Azar ({yc_tr.mean():.2f})")
ejes[1].set(title="Curva precisión-recall (validación cruzada, train)", xlabel="Recall", ylabel="Precisión")
ejes[0].legend(loc="lower right"); ejes[1].legend(loc="upper right")
plt.tight_layout(); plt.show()

# ¿Importa peticiones_especiales (que podría registrarse después de reservar)?
sin_pet = [c for c in columnas_num if c != "peticiones_especiales"]
auc_sin_pet = cross_val_score(Pipeline([("prep", preprocesado(num=sin_pet)), ("clf", LogisticRegression(max_iter=1000))]),
                              Xc_tr[sin_pet + columnas_cat], yc_tr, cv=cv, scoring="roc_auc").mean()
print(f"AUC CV con peticiones_especiales: {tabla_cv_clasif.loc[nombre_logit, 'AUC (CV)']:.3f} · sin ella: {auc_sin_pet:.3f}")

AUC CV con peticiones_especiales: 0.758 · sin ella: 0.755


### 4.3 · Hipótesis de coste (en euros)

> ⚠️ **Estas cifras son una propuesta de partida del cuaderno: el equipo debe revisarlas, discutirlas y firmarlas.**
> Cambiad los valores y la frase de razón si no os convencen: todo lo demás se recalcula.

Referencia de los datos: precio medio ≈ 103 €/noche · estancia media ≈ 5,5 noches (≈ 568 € por reserva).

- **`COSTE_FN` = 150 €.** *Cancelación que no detectamos:* la habitación se queda vacía y solo se consigue revender una parte de las noches; ≈ 1,5 noches medias sin ingresar.
- **`COSTE_FP` = 75 €.** *Falsa alarma:* aceptamos una reserva de más y el cliente sí llega. Reubicarlo cuesta ≈ 1,5 noches medias (≈ 150 €), pero solo ocurre si el hotel está realmente lleno (supuesto: la mitad de las veces) → coste esperado ≈ 75 €.

In [10]:
COSTE_FN = 150.0   # € · cancelación NO detectada (habitación vacía: ≈1,5 noches medias sin ingresar)
COSTE_FP = 75.0    # € · falsa alarma (cliente reubicado ≈150 € × 50 % de veces que el hotel está lleno)

def metricas_umbral(y_real, proba, umbral):
    pred = (np.asarray(proba) >= umbral).astype(int)
    vn, fp, fn, vp = confusion_matrix(y_real, pred, labels=[0, 1]).ravel()
    return {"umbral": umbral, "VN": vn, "FP": fp, "FN": fn, "VP": vp,
            "precision": vp / (vp + fp) if (vp + fp) else np.nan, "recall": vp / (vp + fn)}

def coste_total(y_real, proba, umbral, coste_fn, coste_fp):
    m = metricas_umbral(y_real, proba, umbral)
    return m["FN"] * coste_fn + m["FP"] * coste_fp

def coste_sin_modelo(y_real, coste_fn):
    """«No hacer nada»: nadie se marca como cancelación → todas las canceladas son FN."""
    return int(np.sum(y_real)) * coste_fn

print(f"COSTE_FN = {COSTE_FN:.0f} € · COSTE_FP = {COSTE_FP:.0f} € · umbral teórico = FP/(FP+FN) = {COSTE_FP/(COSTE_FP+COSTE_FN):.2f}")

COSTE_FN = 150 € · COSTE_FP = 75 € · umbral teórico = FP/(FP+FN) = 0.33


### 4.4 · Umbral que minimiza el coste (validación cruzada sobre train)

In [11]:
umbrales = np.round(np.arange(0.05, 0.96, 0.01), 2)
costes_cv = np.array([coste_total(yc_tr, proba_cv, u, COSTE_FN, COSTE_FP) for u in umbrales])
umbral_elegido = float(umbrales[int(np.argmin(costes_cv))])
n_tr = len(yc_tr)
print(f"Umbral elegido (CV, train): {umbral_elegido:.2f}")
print(f"Coste por 1000 reservas (CV): sin modelo {coste_sin_modelo(yc_tr, COSTE_FN)/n_tr*1000:,.0f} € · "
      f"umbral 0,5 {coste_total(yc_tr, proba_cv, 0.5, COSTE_FN, COSTE_FP)/n_tr*1000:,.0f} € · "
      f"umbral {umbral_elegido:.2f}: {costes_cv.min()/n_tr*1000:,.0f} €")

plt.figure(figsize=(8, 4))
plt.plot(umbrales, costes_cv / n_tr * 1000, color="#2a6f97", lw=2)
plt.axvline(umbral_elegido, color="#e63946", ls="--", label=f"Mínimo: {umbral_elegido:.2f}")
plt.axvline(0.5, color="gray", ls=":", label="Umbral por defecto: 0,50")
plt.title(f"Coste según el umbral (FN {COSTE_FN:.0f} € · FP {COSTE_FP:.0f} €)")
plt.xlabel("Umbral de decisión"); plt.ylabel("Coste por 1000 reservas (€)"); plt.legend()
plt.show()

tabla_umbral_cv = pd.DataFrame([{**metricas_umbral(yc_tr, proba_cv, u),
                                 "coste_por_1000 (€)": coste_total(yc_tr, proba_cv, u, COSTE_FN, COSTE_FP) / n_tr * 1000}
                                for u in [0.3, 0.4, 0.5, umbral_elegido]]).round(3)
tabla_umbral_cv[["umbral", "recall", "precision", "FN", "FP", "coste_por_1000 (€)"]]

Umbral elegido (CV, train): 0.32
Coste por 1000 reservas (CV): sin modelo 68,844 € · umbral 0,5 36,078 € · umbral 0.32: 30,562 €


,umbral,recall,precision,FN,FP,coste_por_1000 (€)
0,0.30,0.886,0.569,251,1481,30984.375
1,0.40,0.774,0.623,498,1032,31687.500
2,0.50,0.628,0.673,819,671,36078.125
3,0.32,0.869,0.581,288,1380,30562.500


### 4.5 · Análisis de sensibilidad

¿Qué pasa si los costes cambian? Para cada escenario se vuelve a buscar el umbral óptimo (en validación cruzada, **sin tocar el test**).
Se compara con «no hacer nada» y con el umbral 0,5.

In [12]:
escenarios = {
    "Base":              (COSTE_FN,       COSTE_FP),
    "FN el doble":       (COSTE_FN * 2,   COSTE_FP),
    "FN la mitad":       (COSTE_FN / 2,   COSTE_FP),
    "FP el doble":       (COSTE_FN,       COSTE_FP * 2),
    "FP la mitad":       (COSTE_FN,       COSTE_FP / 2),
    "Ambos el doble":    (COSTE_FN * 2,   COSTE_FP * 2),
}
filas = []
for nombre, (cfn, cfp) in escenarios.items():
    cs = np.array([coste_total(yc_tr, proba_cv, u, cfn, cfp) for u in umbrales])
    u_opt = float(umbrales[int(np.argmin(cs))])
    sin = coste_sin_modelo(yc_tr, cfn); c05 = coste_total(yc_tr, proba_cv, 0.5, cfn, cfp)
    filas.append({"escenario": nombre, "COSTE_FN": cfn, "COSTE_FP": cfp, "umbral_óptimo": u_opt,
                  "sin modelo (€/1000)": sin / n_tr * 1000, "umbral 0,5 (€/1000)": c05 / n_tr * 1000,
                  "umbral óptimo (€/1000)": cs.min() / n_tr * 1000,
                  "ahorro vs sin modelo (€/1000)": (sin - cs.min()) / n_tr * 1000})
sensibilidad = pd.DataFrame(filas).set_index("escenario").round(2)
sensibilidad

,COSTE_FN,COSTE_FP,umbral_óptimo,sin modelo (€/1000),"umbral 0,5 (€/1000)",umbral óptimo (€/1000),ahorro vs sin modelo (€/1000)
escenario,,,,,,,
Base,150.0,75.0,0.32,68843.75,36078.12,30562.50,38281.25
FN el doble,300.0,75.0,0.17,137687.50,61671.88,36046.88,101640.62
FN la mitad,75.0,75.0,0.49,34421.88,23281.25,23171.88,11250.00
FP el doble,150.0,150.0,0.49,68843.75,46562.50,46343.75,22500.00
FP la mitad,150.0,37.5,0.17,68843.75,30835.94,18023.44,50820.31
Ambos el doble,300.0,150.0,0.32,137687.50,72156.25,61125.00,76562.50


In [13]:
fig, ax = plt.subplots(figsize=(8, 4))
for nombre, (cfn, cfp) in escenarios.items():
    cs = np.array([coste_total(yc_tr, proba_cv, u, cfn, cfp) for u in umbrales]) / n_tr * 1000
    ax.plot(umbrales, cs, label=f"{nombre} (óptimo {sensibilidad.loc[nombre, 'umbral_óptimo']:.2f})")
ax.set(title="Coste según el umbral en cada escenario de costes", xlabel="Umbral", ylabel="Coste por 1000 reservas (€)")
ax.legend(fontsize=8); plt.tight_layout(); plt.show()

### 4.6 · Evaluación final en TEST (una sola vez)

Aquí se usa `test` por primera y única vez: se ajusta cada modelo con todo *train* y se miden la matriz de confusión,
AUC, AP y el coste **sin modelo**, **con umbral 0,5** y **con el umbral elegido** (que salió de validación cruzada).

In [14]:
logit_final = Pipeline([("prep", preprocesado()), ("clf", LogisticRegression(max_iter=1000))]).fit(Xc_tr, yc_tr)
proba_test = logit_final.predict_proba(Xc_te)[:, 1]
base_c.fit(Xc_tr, yc_tr)
knn.fit(Xc_tr, yc_tr)
proba_knn_test = knn.predict_proba(Xc_te)[:, 1]
n_te = len(yc_te)

# Métricas con umbral 0,5 (matrices de confusión) y curvas
fig, ejes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (titulo, pred) in zip(ejes, [("Baseline: siempre «no cancela»", base_c.predict(Xc_te)),
                                      ("Logística · umbral 0,5", (proba_test >= 0.5).astype(int)),
                                      (f"Logística · umbral {umbral_elegido:.2f}", (proba_test >= umbral_elegido).astype(int))]):
    ConfusionMatrixDisplay.from_predictions(yc_te, pred, ax=ax, display_labels=["Se mantiene", "Cancela"], cmap="Blues", colorbar=False)
    ax.set_title(titulo); ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
plt.tight_layout(); plt.show()

auc_test = roc_auc_score(yc_te, proba_test); ap_test = average_precision_score(yc_te, proba_test)
auc_knn_test = roc_auc_score(yc_te, proba_knn_test); ap_knn_test = average_precision_score(yc_te, proba_knn_test)
print(f"TEST · logística: AUC {auc_test:.3f} · AP {ap_test:.3f}  |  k-NN: AUC {auc_knn_test:.3f} · AP {ap_knn_test:.3f}")

# Costes en euros (test)
c_sin = coste_sin_modelo(yc_te, COSTE_FN)
c_05 = coste_total(yc_te, proba_test, 0.5, COSTE_FN, COSTE_FP)
c_opt = coste_total(yc_te, proba_test, umbral_elegido, COSTE_FN, COSTE_FP)
m_05 = metricas_umbral(yc_te, proba_test, 0.5); m_opt = metricas_umbral(yc_te, proba_test, umbral_elegido)
resumen_test = pd.DataFrame({
    "coste total test (€)": [c_sin, c_05, c_opt],
    "coste por 1000 reservas (€)": [c_sin / n_te * 1000, c_05 / n_te * 1000, c_opt / n_te * 1000],
    "ahorro vs sin modelo (€)": [0, c_sin - c_05, c_sin - c_opt],
}, index=["Sin modelo (no hacer nada)", "Logística · umbral 0,5", f"Logística · umbral {umbral_elegido:.2f}"]).round(0)
display(resumen_test)
print(f"Con el umbral {umbral_elegido:.2f}: recall {m_opt['recall']:.0%} · precisión {m_opt['precision']:.0%} · "
      f"FN {m_opt['FN']} · FP {m_opt['FP']}")

# Ahorro por temporada (extra: overbooking por temporada)
alta_te = Xc_te["temporada_alta"].values
filas = []
for etiqueta, mask in [("Temporada baja", alta_te == 0), ("Temporada alta", alta_te == 1)]:
    yt, pt = yc_te.values[mask], proba_test[mask]
    sin = coste_sin_modelo(yt, COSTE_FN); con = coste_total(yt, pt, umbral_elegido, COSTE_FN, COSTE_FP)
    filas.append({"temporada": etiqueta, "reservas": int(mask.sum()), "tasa_cancelación": yt.mean(),
                  "ahorro por 1000 reservas (€)": (sin - con) / mask.sum() * 1000})
ahorro_temporada = pd.DataFrame(filas).set_index("temporada").round(3)
ahorro_temporada

TEST · logística: AUC 0.746 · AP 0.729  |  k-NN: AUC 0.727 · AP 0.704


,coste total test (€),coste por 1000 reservas (€),ahorro vs sin modelo (€)
Sin modelo (no hacer nada),82650.0,68875.0,0.0
"Logística · umbral 0,5",45000.0,37500.0,37650.0
Logística · umbral 0.32,38775.0,32312.0,43875.0


Con el umbral 0.32: recall 85% · precisión 57% · FN 81 · FP 355


,reservas,tasa_cancelación,ahorro por 1000 reservas (€)
temporada,,,
Temporada baja,599,0.427,31427.379
Temporada alta,601,0.491,41680.532


### 4.7 · IA responsable: ¿a quién perjudican las falsas alarmas?

Un **falso positivo** es un cliente que va a venir pero el modelo avisa de que cancelará: con *overbooking* es el que
puede acabar **reubicado**. Se mide la tasa de falsos positivos por canal y por país con las probabilidades de validación cruzada de *train*
(para no volver a usar el test) y el umbral elegido.

In [15]:
pred_cv = (proba_cv >= umbral_elegido).astype(int)
aux = Xc_tr.assign(real=yc_tr.values, pred=pred_cv)
aux["pais_origen"] = aux["pais_origen"].fillna("desconocido")

def tabla_fp(col):
    g = aux.groupby(col)
    t = pd.DataFrame({
        "reservas": g.size(),
        "se_marcan_como_cancelación": g["pred"].mean(),
        "tasa_FP (de los que sí vienen)": g.apply(lambda d: ((d["pred"] == 1) & (d["real"] == 0)).sum() / max((d["real"] == 0).sum(), 1)),
        "tasa_FN (de los que cancelan)": g.apply(lambda d: ((d["pred"] == 0) & (d["real"] == 1)).sum() / max((d["real"] == 1).sum(), 1)),
    }).round(3)
    return t.sort_values("tasa_FP (de los que sí vienen)", ascending=False)

fp_canal, fp_pais = tabla_fp("canal"), tabla_fp("pais_origen")
display(fp_canal); display(fp_pais)

fig, ejes = plt.subplots(1, 2, figsize=(12, 3.8))
fp_canal["tasa_FP (de los que sí vienen)"].plot(kind="bar", ax=ejes[0], color="#2a6f97", rot=20)
ejes[0].set(title="Tasa de falsos positivos por canal", ylabel="% de clientes que vienen y se marcan")
fp_pais["tasa_FP (de los que sí vienen)"].plot(kind="bar", ax=ejes[1], color="#2a6f97", rot=0)
ejes[1].set(title="Tasa de falsos positivos por país", ylabel="% de clientes que vienen y se marcan")
plt.tight_layout(); plt.show()

brecha_canal = fp_canal["tasa_FP (de los que sí vienen)"].max() - fp_canal["tasa_FP (de los que sí vienen)"].min()
brecha_pais = fp_pais["tasa_FP (de los que sí vienen)"].max() - fp_pais["tasa_FP (de los que sí vienen)"].min()
print(f"Brecha de tasa FP entre canales: {brecha_canal:.1%} · entre países: {brecha_pais:.1%}")

,reservas,se_marcan_como_cancelación,tasa_FP (de los que sí vienen),tasa_FN (de los que cancelan)
canal,,,,
web_intermediaria,2437,0.829,0.707,0.067
agencia,947,0.671,0.525,0.144
empresa,260,0.627,0.500,0.205
directo,1156,0.413,0.293,0.324


,reservas,se_marcan_como_cancelación,tasa_FP (de los que sí vienen),tasa_FN (de los que cancelan)
pais_origen,,,,
desconocido,107,0.720,0.636,0.146
Nordicos,333,0.718,0.578,0.118
UK,1197,0.713,0.567,0.127
IT,229,0.712,0.546,0.109
otros,467,0.672,0.518,0.138
NL,285,0.649,0.515,0.167
DE,711,0.677,0.513,0.120
ES,1298,0.668,0.509,0.144
FR,173,0.665,0.429,0.073


Brecha de tasa FP entre canales: 41.4% · entre países: 20.7%


## 5 · Cifras del informe (qué celda produce cada una)

Esta celda reúne **todas las cifras** que aparecen en `informe_modelado.md` y las guarda en `cifras_informe.json`.
Así se puede comprobar que informe y cuaderno coinciden.

In [16]:
cifras = {
    # --- Precio (celdas 3.x)
    "precio_baseline_MAE": tabla_precio.loc["baseline (precio medio)", "MAE (€)"],
    "precio_baseline_RMSE": tabla_precio.loc["baseline (precio medio)", "RMSE (€)"],
    "precio_lineal_MAE": tabla_precio.loc["regresión lineal", "MAE (€)"],
    "precio_lineal_RMSE": tabla_precio.loc["regresión lineal", "RMSE (€)"],
    "precio_lineal_R2": tabla_precio.loc["regresión lineal", "R²"],
    "precio_inter_MAE": tabla_precio.loc["lineal + interacciones", "MAE (€)"],
    "precio_inter_RMSE": tabla_precio.loc["lineal + interacciones", "RMSE (€)"],
    "precio_inter_R2": tabla_precio.loc["lineal + interacciones", "R²"],
    "precio_mejora_MAE": mejora_mae_precio,
    "precio_base_estandar_baja": float(lin_p.intercept_),
    "precio_extra_suite": float(coefs["tipo_habitacion_suite"]),
    "precio_extra_familiar": float(coefs["tipo_habitacion_familiar"]),
    "precio_extra_superior": float(coefs["tipo_habitacion_superior"]),
    "precio_extra_temporada_alta": float(coefs["temporada_alta"]),
    # --- Cancelación (4.2)
    "canc_tasa_train": float(yc_tr.mean()), "mejor_k": int(mejor_k),
    "canc_AUC_cv_logit": tabla_cv_clasif.loc[nombre_logit, "AUC (CV)"], "canc_AP_cv_logit": tabla_cv_clasif.loc[nombre_logit, "AP (CV)"],
    "canc_AUC_cv_knn": tabla_cv_clasif.loc[nombre_knn, "AUC (CV)"], "canc_AP_cv_knn": tabla_cv_clasif.loc[nombre_knn, "AP (CV)"],
    "canc_AUC_cv_sin_peticiones": auc_sin_pet,
    # --- Costes y umbral (4.3-4.4)
    "COSTE_FN": COSTE_FN, "COSTE_FP": COSTE_FP, "umbral_elegido": umbral_elegido,
    # --- Test final (4.6)
    "test_AUC_logit": auc_test, "test_AP_logit": ap_test, "test_AUC_knn": auc_knn_test, "test_AP_knn": ap_knn_test,
    "test_n": int(n_te), "test_n_canceladas": int(yc_te.sum()),
    "test_coste_sin_modelo": float(c_sin), "test_coste_umbral05": float(c_05), "test_coste_umbral_elegido": float(c_opt),
    "test_ahorro_vs_sin_modelo": float(c_sin - c_opt), "test_ahorro_vs_umbral05": float(c_05 - c_opt),
    "test_ahorro_por_1000": float((c_sin - c_opt) / n_te * 1000),
    "test_recall_elegido": float(m_opt["recall"]), "test_precision_elegido": float(m_opt["precision"]),
    "test_FN_elegido": int(m_opt["FN"]), "test_FP_elegido": int(m_opt["FP"]),
    "test_FN_05": int(m_05["FN"]), "test_FP_05": int(m_05["FP"]),
    "test_recall_05": float(m_05["recall"]), "test_precision_05": float(m_05["precision"]),
    "test_acc_baseline": float(accuracy_score(yc_te, base_c.predict(Xc_te))),
    "test_ahorro_baja_por_1000": float(ahorro_temporada.loc["Temporada baja", "ahorro por 1000 reservas (€)"]),
    "test_ahorro_alta_por_1000": float(ahorro_temporada.loc["Temporada alta", "ahorro por 1000 reservas (€)"]),
    # --- IA responsable (4.7)
    "brecha_FP_canal": float(brecha_canal), "brecha_FP_pais": float(brecha_pais),
    "canal_FP_max": fp_canal.index[0], "canal_FP_min": fp_canal.index[-1],
    "pais_FP_max": fp_pais.index[0], "pais_FP_min": fp_pais.index[-1],
    "FP_canal_max_valor": float(fp_canal.iloc[0, 2]), "FP_canal_min_valor": float(fp_canal.iloc[-1, 2]),
    "FP_pais_max_valor": float(fp_pais.iloc[0, 2]), "FP_pais_min_valor": float(fp_pais.iloc[-1, 2]),
}
cifras = {k: (round(v, 4) if isinstance(v, float) else v) for k, v in cifras.items()}
with open("cifras_informe.json", "w", encoding="utf-8") as f:
    json.dump(cifras, f, ensure_ascii=False, indent=2)
pd.Series(cifras)

precio_baseline_MAE                       25.564
precio_baseline_RMSE                      33.328
precio_lineal_MAE                          7.239
precio_lineal_RMSE                         9.704
precio_lineal_R2                           0.915
precio_inter_MAE                           6.758
precio_inter_RMSE                          8.936
precio_inter_R2                            0.928
precio_mejora_MAE                          0.481
precio_base_estandar_baja                67.2705
precio_extra_suite                         101.2
precio_extra_familiar                       45.0
precio_extra_superior                       28.4
precio_extra_temporada_alta                 22.4
canc_tasa_train                            0.459
mejor_k                                      201
canc_AUC_cv_logit                          0.758
canc_AP_cv_logit                           0.719
canc_AUC_cv_knn                            0.734
canc_AP_cv_knn                             0.689
canc_AUC_cv_sin_peti

In [17]:
# Sensibilidad (para copiar al informe)
print(sensibilidad[["COSTE_FN", "COSTE_FP", "umbral_óptimo", "ahorro vs sin modelo (€/1000)"]].to_string())
print()
print("Tabla de precios:\n", tarifa_ref.to_string())

                COSTE_FN  COSTE_FP  umbral_óptimo  ahorro vs sin modelo (€/1000)
escenario                                                                       
Base               150.0      75.0           0.32                       38281.25
FN el doble        300.0      75.0           0.17                      101640.62
FN la mitad         75.0      75.0           0.49                       11250.00
FP el doble        150.0     150.0           0.49                       22500.00
FP la mitad        150.0      37.5           0.17                       50820.31
Ambos el doble     300.0     150.0           0.32                       76562.50

Tabla de precios:
 temporada   alta   baja  recargo_alta (€)
tipo                                     
estandar    87.0   70.0              17.0
superior   119.0   95.0              24.0
familiar   137.0  110.0              27.0
suite      198.0  161.0              37.0
